# Tugas Praktikum - Support Vector Machine

**Nama:** Radit Firansah  
**NIM:** 244107020196

Klasifikasi gender dari voice.csv dan klasifikasi siang/malam dengan fitur histogram menggunakan SVM.

## Bagian 1 - Klasifikasi Suara Male/Female dengan voice.csv

Pada bagian ini kita membangun model SVM dengan tiga kernel (linear, polynomial, RBF) dan dua rasio split (70:30 dan 80:20), lalu menabulasikan akurasinya.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

Membaca dataset voice.csv

In [2]:
df = pd.read_csv('../data/voice.csv')
print(df.shape)
df.head()

(3168, 21)


,meanfreq,sd,median,Q25,Q75,IQR,skew,kurt,sp.ent,sfm,...,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label
0,0.059781,0.064241,0.032027,0.015071,0.090193,0.075122,12.863462,274.402906,0.893369,0.491918,...,0.059781,0.084279,0.015702,0.275862,0.007812,0.007812,0.007812,0.000000,0.000000,male
1,0.066009,0.067310,0.040229,0.019414,0.092666,0.073252,22.423285,634.613855,0.892193,0.513724,...,0.066009,0.107937,0.015826,0.250000,0.009014,0.007812,0.054688,0.046875,0.052632,male
2,0.077316,0.083829,0.036718,0.008701,0.131908,0.123207,30.757155,1024.927705,0.846389,0.478905,...,0.077316,0.098706,0.015656,0.271186,0.007990,0.007812,0.015625,0.007812,0.046512,male
3,0.151228,0.072111,0.158011,0.096582,0.207955,0.111374,1.232831,4.177296,0.963322,0.727232,...,0.151228,0.088965,0.017798,0.250000,0.201497,0.007812,0.562500,0.554688,0.247119,male
4,0.135120,0.079146,0.124656,0.078720,0.206045,0.127325,1.101174,4.333713,0.971955,0.783568,...,0.135120,0.106398,0.016931,0.266667,0.712812,0.007812,5.484375,5.476562,0.208274,male


Memisahkan fitur (X) dan label (y)

In [3]:
X = df.drop('label', axis=1).values
y = df['label'].values
print(X.shape, y.shape)

(3168, 20) (3168,)


Melatih model untuk setiap kombinasi split dan kernel, lalu menyimpan akurasinya ke dalam tabel.

In [4]:
hasil = pd.DataFrame(index=['linear', 'poly', 'rbf'], columns=['70:30', '80:20'])

for test_size, nama_split in [(0.3, '70:30'), (0.2, '80:20')]:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    for kernel in ['linear', 'poly', 'rbf']:
        model = SVC(kernel=kernel)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        hasil.loc[kernel, nama_split] = accuracy_score(y_test, y_pred)

hasil

,70:30,80:20
linear,0.97897,0.974763
poly,0.960042,0.955836
rbf,0.983176,0.98265


### Analisa Bagian 1

Tabel di atas menunjukkan akurasi tiap kernel pada masing-masing rasio split. Kernel linear dan RBF umumnya memberikan akurasi yang tinggi karena fitur akustik suara sudah cukup memisahkan kelas male dan female secara (hampir) linear. Kernel polynomial perlu penyetelan derajat dan parameter lain agar optimal. Perbedaan rasio 70:30 dan 80:20 relatif kecil karena jumlah data cukup banyak (3168 baris).

## Bagian 2 - Klasifikasi Siang/Malam dengan Fitur Histogram

Pada bagian ini kita menggunakan citra siang/malam, mengekstrak fitur histogram dari citra grayscale, lalu mengklasifikasikan dengan SVM kernel RBF.

In [5]:
import glob
import cv2

Membaca semua gambar training, lalu membuat fitur histogram grayscale (32 bin, dinormalisasi) dan label (day=1, night=0).

In [6]:
files = glob.glob('../data/images/training/*/*.jpg')
print('Jumlah gambar:', len(files))

X_hist = []
y_hist = []

for f in files:
    img = cv2.imread(f)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hist = cv2.calcHist([gray], [0], None, [32], [0, 256]).flatten()
    hist = hist / hist.sum()
    X_hist.append(hist)

    label = f.split('/')[-2]
    y_hist.append(1 if label == 'day' else 0)

X_hist = np.array(X_hist)
y_hist = np.array(y_hist)
print(X_hist.shape, y_hist.shape)

Jumlah gambar: 240


(240, 32) (240,)


Membagi data 80:20, menstandarkan fitur, lalu mencoba beberapa set hyperparameter RBF.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_hist, y_hist, test_size=0.2, random_state=42, stratify=y_hist)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

for C, gamma in [(1, 'scale'), (10, 'scale'), (10, 0.01)]:
    model = SVC(kernel='rbf', C=C, gamma=gamma)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f'C={C}, gamma={gamma} -> akurasi: {acc:.4f}')

C=1, gamma=scale -> akurasi: 0.9792
C=10, gamma=scale -> akurasi: 0.9792
C=10, gamma=0.01 -> akurasi: 0.9792


### Analisa Bagian 2

Fitur histogram grayscale menangkap sebaran intensitas cahaya pada citra sehingga mampu membedakan siang (citra lebih terang) dan malam (citra lebih gelap). Dengan SVM kernel RBF, akurasi yang diperoleh cukup tinggi. Nilai `C` dan `gamma` memengaruhi batas keputusan: `C` besar membuat model lebih ketat terhadap data latih, sedangkan `gamma` mengatur seberapa luas pengaruh satu titik data. Penyetelan hyperparameter yang tepat dapat meningkatkan akurasi.

## Kesimpulan

1. SVM dapat menangani data linier maupun non-linier dengan pemilihan kernel yang tepat (linear, polynomial, RBF).
2. Pada dataset voice.csv, kernel linear dan RBF memberikan performa terbaik untuk klasifikasi male/female.
3. Pada citra siang/malam, fitur sederhana berupa histogram grayscale sudah cukup baik untuk dipisahkan oleh SVM RBF.
4. Pemilihan hyperparameter (C dan gamma) serta skala fitur (StandardScaler) sangat berpengaruh terhadap performa model.